# COSC2671 — Assignment 2: Data Collection
## What Makes a YouTube Gaming Video Go Viral?

This notebook collects data from the YouTube Data API v3 for three gaming niches:
- **Minecraft**
- **Valorant**
- **GTA VI**

For each niche we collect:
1. Video metadata (views, likes, comments, duration, tags, etc.)
2. Top-level comments + replies per video
3. Channel subscriber counts (for normalising virality)

**Output files:**
- `data/videos.csv` — all video metadata
- `data/comments.csv` — all comments and replies
- `data/channels.csv` — channel subscriber info

**API Quota:** YouTube Data API v3 free tier = 10,000 units/day.  
- search.list = 100 units per call  
- videos.list = 1 unit per call  
- commentThreads.list = 1 unit per call  


## Cell 1 — Install Dependencies

In [ ]:
# Run once to install required packages
#!pip install google-api-python-client python-dotenv pandas isodate tqdm

## Cell 2 — Imports & Configuration

In [2]:
import os
import time
import json
import isodate
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm import tqdm

# ── Load API key from .env file ───────────────────────────────────────────────
load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')

if not API_KEY:
    raise ValueError("API key not found. Make sure .env file exists with YOUTUBE_API_KEY set.")

print("API key loaded successfully.")

# ── Build YouTube API client ──────────────────────────────────────────────────
youtube = build('youtube', 'v3', developerKey=API_KEY)

# ── Configuration ─────────────────────────────────────────────────────────────
# Gaming niches to collect data for
NICHES = {
    'minecraft': 'Minecraft',
    'valorant': 'Valorant gameplay',
    'gta6': 'GTA 6 trailer'
}

# Max videos per niche (50 per page × pages)
# 200 videos × 3 niches = 600 videos total
MAX_VIDEOS_PER_NICHE = 200

# Max comments per video (to stay within API quota)
MAX_COMMENTS_PER_VIDEO = 100

# Output directory
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Delay between API calls (seconds) to avoid rate limiting
API_DELAY = 0.3

print(f"Configuration ready. Output directory: {OUTPUT_DIR}/")
print(f"Niches: {list(NICHES.keys())}")
print(f"Max videos per niche: {MAX_VIDEOS_PER_NICHE}")

API key loaded successfully.
Configuration ready. Output directory: data/
Niches: ['minecraft', 'valorant', 'gta6']
Max videos per niche: 200


## Cell 3 — Helper Functions

In [3]:
def parse_duration(iso_duration):
    """
    Convert ISO 8601 duration string (e.g. 'PT12M34S') to seconds.
    Returns 0 if parsing fails.
    """
    try:
        return int(isodate.parse_duration(iso_duration).total_seconds())
    except Exception:
        return 0


def safe_int(value, default=0):
    """Safely convert a value to int, returning default if conversion fails."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def search_videos(query, max_results=200):
    """
    Search YouTube for videos matching a query.
    Returns a list of video IDs.
    
    API cost: 100 units per search.list call
    Each page returns up to 50 results.
    """
    video_ids = []
    next_page_token = None
    pages_fetched = 0
    max_pages = max_results // 50 + 1

    print(f"  Searching for: '{query}'")

    while len(video_ids) < max_results and pages_fetched < max_pages:
        try:
            params = {
                'q': query,
                'part': 'id',
                'type': 'video',
                'maxResults': 50,
                'order': 'relevance',          # Mix of popular and relevant
                'relevanceLanguage': 'en',
                'videoDuration': 'medium',     # 4–20 minutes (typical gaming content)
            }
            if next_page_token:
                params['pageToken'] = next_page_token

            response = youtube.search().list(**params).execute()
            items = response.get('items', [])
            ids = [item['id']['videoId'] for item in items if item['id'].get('videoId')]
            video_ids.extend(ids)

            next_page_token = response.get('nextPageToken')
            pages_fetched += 1

            if not next_page_token:
                break

            time.sleep(API_DELAY)

        except HttpError as e:
            print(f"  HTTP error during search: {e}")
            break

    print(f"  Found {len(video_ids)} video IDs")
    return video_ids[:max_results]


def get_video_details(video_ids):
    """
    Fetch full metadata for a list of video IDs.
    Processes in batches of 50 (API maximum).
    
    Returns a list of dicts with video metadata.
    API cost: 1 unit per videos.list call (up to 50 IDs per call)
    """
    all_videos = []
    batch_size = 50

    for i in range(0, len(video_ids), batch_size):
        batch = video_ids[i:i + batch_size]
        try:
            response = youtube.videos().list(
                part='snippet,statistics,contentDetails',
                id=','.join(batch)
            ).execute()

            for item in response.get('items', []):
                snippet = item.get('snippet', {})
                stats = item.get('statistics', {})
                content = item.get('contentDetails', {})

                video = {
                    'video_id': item['id'],
                    'title': snippet.get('title', ''),
                    'channel_id': snippet.get('channelId', ''),
                    'channel_title': snippet.get('channelTitle', ''),
                    'published_at': snippet.get('publishedAt', ''),
                    'description': snippet.get('description', '')[:500],  # truncate
                    'tags': '|'.join(snippet.get('tags', [])),
                    'category_id': snippet.get('categoryId', ''),
                    'view_count': safe_int(stats.get('viewCount')),
                    'like_count': safe_int(stats.get('likeCount')),
                    'comment_count': safe_int(stats.get('commentCount')),
                    'duration_seconds': parse_duration(content.get('duration', 'PT0S')),
                    'definition': content.get('definition', ''),  # hd or sd
                    'caption': content.get('caption', 'false'),
                    'collected_at': datetime.utcnow().isoformat()
                }
                all_videos.append(video)

            time.sleep(API_DELAY)

        except HttpError as e:
            print(f"  HTTP error fetching video details (batch {i}): {e}")
            continue

    return all_videos


def get_comments(video_id, max_comments=100):
    """
    Fetch top-level comments and replies for a single video.
    
    Returns a list of dicts.
    API cost: 1 unit per commentThreads.list call
    """
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        try:
            params = {
                'part': 'snippet,replies',
                'videoId': video_id,
                'maxResults': 100,
                'order': 'relevance',  # Top comments first
                'textFormat': 'plainText'
            }
            if next_page_token:
                params['pageToken'] = next_page_token

            response = youtube.commentThreads().list(**params).execute()

            for thread in response.get('items', []):
                top = thread['snippet']['topLevelComment']['snippet']

                # Top-level comment
                comments.append({
                    'video_id': video_id,
                    'comment_id': thread['snippet']['topLevelComment']['id'],
                    'parent_id': None,
                    'author_channel_id': top.get('authorChannelId', {}).get('value', ''),
                    'author_display_name': top.get('authorDisplayName', ''),
                    'text': top.get('textDisplay', ''),
                    'like_count': safe_int(top.get('likeCount')),
                    'reply_count': safe_int(thread['snippet'].get('totalReplyCount')),
                    'published_at': top.get('publishedAt', ''),
                    'is_reply': False
                })

                # Replies to this comment
                if thread.get('replies'):
                    for reply in thread['replies']['comments']:
                        rs = reply['snippet']
                        comments.append({
                            'video_id': video_id,
                            'comment_id': reply['id'],
                            'parent_id': thread['snippet']['topLevelComment']['id'],
                            'author_channel_id': rs.get('authorChannelId', {}).get('value', ''),
                            'author_display_name': rs.get('authorDisplayName', ''),
                            'text': rs.get('textDisplay', ''),
                            'like_count': safe_int(rs.get('likeCount')),
                            'reply_count': 0,
                            'published_at': rs.get('publishedAt', ''),
                            'is_reply': True
                        })

                if len(comments) >= max_comments:
                    break

            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break

            time.sleep(API_DELAY)

        except HttpError as e:
            # Comments may be disabled on some videos — skip silently
            if 'commentsDisabled' in str(e) or '403' in str(e):
                break
            print(f"  HTTP error fetching comments for {video_id}: {e}")
            break

    return comments


def get_channel_details(channel_ids):
    """
    Fetch subscriber counts and video counts for a list of channel IDs.
    Processes in batches of 50.
    """
    channels = []
    unique_ids = list(set(channel_ids))
    batch_size = 50

    for i in range(0, len(unique_ids), batch_size):
        batch = unique_ids[i:i + batch_size]
        try:
            response = youtube.channels().list(
                part='snippet,statistics',
                id=','.join(batch)
            ).execute()

            for item in response.get('items', []):
                stats = item.get('statistics', {})
                channels.append({
                    'channel_id': item['id'],
                    'channel_title': item['snippet'].get('title', ''),
                    'subscriber_count': safe_int(stats.get('subscriberCount')),
                    'video_count': safe_int(stats.get('videoCount')),
                    'view_count_total': safe_int(stats.get('viewCount'))
                })

            time.sleep(API_DELAY)

        except HttpError as e:
            print(f"  HTTP error fetching channel details: {e}")
            continue

    return channels


print("Helper functions defined.")

Helper functions defined.


## Cell 4 — Collect Video Metadata for All Niches

In [4]:
all_videos = []

for niche_key, query in NICHES.items():
    print(f"\n{'='*60}")
    print(f"Niche: {niche_key.upper()}")
    print(f"{'='*60}")

    # Step 1: Search for video IDs
    video_ids = search_videos(query, max_results=MAX_VIDEOS_PER_NICHE)

    # Step 2: Fetch full metadata for those IDs
    print(f"  Fetching metadata for {len(video_ids)} videos...")
    videos = get_video_details(video_ids)

    # Tag each video with its niche
    for v in videos:
        v['niche'] = niche_key

    all_videos.extend(videos)
    print(f"  Collected {len(videos)} videos for niche '{niche_key}'")

# Save to CSV
videos_df = pd.DataFrame(all_videos)
videos_df.drop_duplicates(subset='video_id', inplace=True)
videos_df.to_csv(f'{OUTPUT_DIR}/videos.csv', index=False)

print(f"\n{'='*60}")
print(f"Total videos collected: {len(videos_df)}")
print(f"Saved to: {OUTPUT_DIR}/videos.csv")
videos_df.head(3)


Niche: MINECRAFT
  Searching for: 'Minecraft'
  Found 200 video IDs
  Fetching metadata for 200 videos...
  Collected 200 videos for niche 'minecraft'

Niche: VALORANT
  Searching for: 'Valorant gameplay'
  Found 200 video IDs
  Fetching metadata for 200 videos...
  Collected 200 videos for niche 'valorant'

Niche: GTA6
  Searching for: 'GTA 6 trailer'
  Found 200 video IDs
  Fetching metadata for 200 videos...
  Collected 200 videos for niche 'gta6'

Total videos collected: 558
Saved to: data/videos.csv


,video_id,title,channel_id,channel_title,published_at,description,tags,category_id,view_count,like_count,comment_count,duration_seconds,definition,caption,collected_at,niche
0,tOp-stUCFEQ,Is Farzy Quitting His Hardcore World? - Farzy Q&A,UCY9l-w7ePGbLWkXXzJki1DA,Farzy 2,2026-05-10T14:30:41Z,👀 Main Channel: https://www.youtube.com/FarzyM...,farzy|farzy minecraft|minecraft hardcore|farzy...,20,19972,853,192,838,hd,false,2026-05-12T16:07:22.342847,minecraft
1,HgFtfiAhMPE,We Simulate Civilization in Minecraft,UC0eLBYhxW9HC0P9PXQ73mpQ,Cash,2026-05-10T16:00:35Z,Instagram: https://www.instagram.com/cashmarco...,Cash|Nico|Nico and Cash|Cash and Nico|Minecraf...,24,489863,6629,1448,922,hd,false,2026-05-12T16:07:22.342847,minecraft
2,_uQjoDgiz5I,You Meet MY CRAZY FAN GIRLS On ONE BLOCK in Mi...,UClCvmntMJCrerMDSc1WQGNg,Omz,2026-05-11T05:59:20Z,Today You Meets MY CRAZY FAN GIRLS On ONE BLOC...,Cash|Nico|Nico and Cash|Cash and Nico|Minecraf...,20,301625,5446,1055,1138,hd,false,2026-05-12T16:07:22.342847,minecraft


## Cell 5 — Collect Channel Details

In [5]:
print("Fetching channel subscriber counts...")

channel_ids = videos_df['channel_id'].dropna().unique().tolist()
print(f"Unique channels: {len(channel_ids)}")

channels = get_channel_details(channel_ids)
channels_df = pd.DataFrame(channels)
channels_df.to_csv(f'{OUTPUT_DIR}/channels.csv', index=False)

print(f"Channel data saved. Shape: {channels_df.shape}")
channels_df.head(3)

Fetching channel subscriber counts...
Unique channels: 254
Channel data saved. Shape: (254, 5)


,channel_id,channel_title,subscriber_count,video_count,view_count_total
0,UCfIZB4PmH8LPesRVZNjDRxQ,Circus Craft,129000,40,34325117
1,UCmp5y07YIV6i2jPX4x82hVQ,Hollow,5160000,6859,1887454106
2,UCClc972nxK3t9y9J14XjNaw,Krissy Plays,88700,275,9868670


## Cell 6 — Collect Comments for All Videos

In [6]:
# ── Filter to videos that have comments ──────────────────────────────────────
videos_with_comments = videos_df[videos_df['comment_count'] > 0]
print(f"Videos with comments enabled: {len(videos_with_comments)} / {len(videos_df)}")

# ── Collect comments ──────────────────────────────────────────────────────────
all_comments = []
failed_videos = []

for _, row in tqdm(videos_with_comments.iterrows(), total=len(videos_with_comments), desc="Collecting comments"):
    vid = row['video_id']
    niche = row['niche']

    comments = get_comments(vid, max_comments=MAX_COMMENTS_PER_VIDEO)

    if comments:
        for c in comments:
            c['niche'] = niche
        all_comments.extend(comments)
    else:
        failed_videos.append(vid)

    time.sleep(API_DELAY)

# Save to CSV
comments_df = pd.DataFrame(all_comments)
comments_df.drop_duplicates(subset='comment_id', inplace=True)
comments_df.to_csv(f'{OUTPUT_DIR}/comments.csv', index=False)

print(f"\nTotal comments collected: {len(comments_df)}")
print(f"Videos with no accessible comments: {len(failed_videos)}")
print(f"Saved to: {OUTPUT_DIR}/comments.csv")
comments_df.head(3)

Videos with comments enabled: 556 / 558



Total comments collected: 46283
Videos with no accessible comments: 1
Saved to: data/comments.csv


,video_id,comment_id,parent_id,author_channel_id,author_display_name,text,like_count,reply_count,published_at,is_reply,niche
0,tOp-stUCFEQ,UgzoeaCbbK4nym9d8XJ4AaABAg,None,UC02DXK3dVaeL2SxhjyJEBng,@RubyStanding-f9p,You should build a roller coaster,49,6,2026-05-10T14:36:41Z,False,minecraft
1,tOp-stUCFEQ,UgzoeaCbbK4nym9d8XJ4AaABAg.AWcrJSIOnoxAWct1nbZDmu,UgzoeaCbbK4nym9d8XJ4AaABAg,UCbJJmFHzVLzqvJ4kFQkOVzg,@Beckett-b8u,Yeah,5,0,2026-05-10T14:51:45Z,True,minecraft
2,tOp-stUCFEQ,UgzoeaCbbK4nym9d8XJ4AaABAg.AWcrJSIOnoxAWcya0bDkZU,UgzoeaCbbK4nym9d8XJ4AaABAg,UC3FKqSunycj5wO6YBKcJOPw,@turbocreeperx,Yeah,4,0,2026-05-10T15:40:15Z,True,minecraft


## Cell 7 — Add Virality Labels to Videos

In [7]:
# ── Merge channel subscriber counts into videos ───────────────────────────────
videos_df = videos_df.merge(
    channels_df[['channel_id', 'subscriber_count']],
    on='channel_id',
    how='left'
)

# ── Normalised view score ─────────────────────────────────────────────────────
# View count normalised by channel size avoids large channels always dominating
# Small channels with viral videos score higher than expected
videos_df['subscriber_count'] = videos_df['subscriber_count'].fillna(1)
videos_df['normalised_views'] = videos_df['view_count'] / (videos_df['subscriber_count'] + 1)

# ── Virality tier labels (per niche) ──────────────────────────────────────────
# Top 25% = Viral, Bottom 25% = Non-viral, Middle 50% = Mid
def assign_virality_tier(group):
    q75 = group['normalised_views'].quantile(0.75)
    q25 = group['normalised_views'].quantile(0.25)
    group['virality_tier'] = group['normalised_views'].apply(
        lambda x: 'viral' if x >= q75 else ('non_viral' if x <= q25 else 'mid')
    )
    return group

videos_df = videos_df.groupby('niche', group_keys=False).apply(assign_virality_tier)

# Save updated videos with labels
videos_df.to_csv(f'{OUTPUT_DIR}/videos.csv', index=False)

print("Virality tier distribution:")
print(videos_df.groupby(['niche', 'virality_tier']).size().unstack(fill_value=0))
print(f"\nSaved updated videos.csv with virality labels.")

Virality tier distribution:
virality_tier  mid  non_viral  viral
niche                               
gta6            89         46     46
minecraft       98         50     50
valorant        89         45     45

Saved updated videos.csv with virality labels.


C:\Users\mitun\AppData\Local\Temp\ipykernel_20780\641806680.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  videos_df = videos_df.groupby('niche', group_keys=False).apply(assign_virality_tier)


## Cell 8 — Quick Data Quality Check

In [8]:
print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)

print(f"\nVideos collected:")
print(f"  Total: {len(videos_df)}")
for niche in NICHES:
    n = len(videos_df[videos_df['niche'] == niche])
    print(f"  {niche}: {n}")

print(f"\nComments collected:")
print(f"  Total: {len(comments_df)}")
for niche in NICHES:
    n = len(comments_df[comments_df['niche'] == niche])
    print(f"  {niche}: {n}")

top_level = comments_df[comments_df['is_reply'] == False]
replies = comments_df[comments_df['is_reply'] == True]
print(f"\n  Top-level comments: {len(top_level)}")
print(f"  Replies: {len(replies)}")

print(f"\nUnique commenters: {comments_df['author_channel_id'].nunique()}")
print(f"Unique channels: {len(channels_df)}")

print(f"\nMissing values in videos_df:")
print(videos_df.isnull().sum()[videos_df.isnull().sum() > 0])

print(f"\nView count stats (by niche):")
print(videos_df.groupby('niche')['view_count'].describe())

print(f"\n{'='*60}")
print("Data collection complete. Files saved in 'data/' folder.")
print("  - data/videos.csv")
print("  - data/comments.csv")
print("  - data/channels.csv")
print("="*60)

DATA COLLECTION SUMMARY

Videos collected:
  Total: 558
  minecraft: 198
  valorant: 179
  gta6: 181

Comments collected:
  Total: 46283
  minecraft: 17689
  valorant: 13423
  gta6: 15171

  Top-level comments: 28280
  Replies: 18003

Unique commenters: 35726
Unique channels: 254

Missing values in videos_df:
Series([], dtype: int64)

View count stats (by niche):
           count          mean           std     min        25%        50%  \
niche                                                                        
gta6       181.0  4.107431e+05  1.542413e+06   423.0   11165.00    31740.0   
minecraft  198.0  3.239242e+06  8.249898e+06  6016.0  476164.25  1271791.5   
valorant   179.0  3.355494e+05  6.423705e+05  2519.0   43987.00   142561.0   

                 75%         max  
niche                             
gta6        218937.0  18894068.0  
minecraft  2358816.0  85366600.0  
valorant    324573.0   5479705.0  

Data collection complete. Files saved in 'data/' folder.
  - data/v

## Cell 9 — Creating a Representative Sample for Submission

The assignment requires submitting a representative data sample (max 10MB). This cell creates a small sample showing the data structure.

In [ ]:
SAMPLE_DIR = 'data/sample'
os.makedirs(SAMPLE_DIR, exist_ok=True)

# 20 videos per niche (60 total)
video_sample = videos_df.groupby('niche').apply(lambda x: x.sample(min(20, len(x)), random_state=42)).reset_index(drop=True)
video_sample.to_csv(f'{SAMPLE_DIR}/videos_sample.csv', index=False)

# 200 comments per niche (600 total)
sample_video_ids = video_sample['video_id'].tolist()
comment_sample = comments_df[comments_df['video_id'].isin(sample_video_ids)]
comment_sample = comment_sample.groupby('niche').apply(lambda x: x.sample(min(200, len(x)), random_state=42)).reset_index(drop=True)
comment_sample.to_csv(f'{SAMPLE_DIR}/comments_sample.csv', index=False)

# All channel data (already small)
channels_df.to_csv(f'{SAMPLE_DIR}/channels_sample.csv', index=False)

# Check sizes
for fname in os.listdir(SAMPLE_DIR):
    fpath = os.path.join(SAMPLE_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname}: {size_kb:.1f} KB")

print(f"\nSample saved to {SAMPLE_DIR}/")


  channels_sample.csv: 14.3 KB
  comments_sample.csv: 116.8 KB
  videos_sample.csv: 55.2 KB

Sample saved to data/sample/


C:\Users\mitun\AppData\Local\Temp\ipykernel_20780\1851591173.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  video_sample = videos_df.groupby('niche').apply(lambda x: x.sample(min(20, len(x)), random_state=42)).reset_index(drop=True)
C:\Users\mitun\AppData\Local\Temp\ipykernel_20780\1851591173.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comment_sample = comment_sample.groupby('niche').apply(lambda x: x.s